# Sesión 4 — Agrupar

**Fase 1 · Sesión 4** · Cuaderno `05_agrupacion.ipynb`

Hasta ahora sabía **quitar** cosas de la tabla: filtrar filas, elegir columnas, pedir una casilla concreta. Agrupar es otra cosa: **parte la tabla en montones y saca un número de cada montón**. La tabla que sale ya no tiene 344 filas, tiene una fila por grupo.

Es la operación que más se usa en el trabajo real, porque casi ninguna pregunta se hace sobre una fila suelta. Se hacen sobre grupos: ventas por región, clientes por franja de edad, averías por modelo.

In [1]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("penguins")

## Bloque 1 — Qué hace `groupby` por dentro

Siempre lo mismo, aunque no se vea:

| Paso | Qué hace |
|---|---|
| 1. Dividir | Reparte las filas en montones según los valores de la columna |
| 2. Calcular | Saca un número de cada montón (media, suma, cuenta...) |
| 3. Juntar | Pega los resultados en una tabla nueva, un grupo por fila |

**Tip:** los nombres de los grupos no se los digo yo. Pandas mira la columna, ve qué valores distintos hay y monta un grupo por cada uno. Si mañana aparece una especie nueva, se hacen cuatro grupos sin tocar el código.

### El objeto que no enseña nada

`df.groupby("species")` sola **no calcula nada todavía**. Solo ha hecho el paso 1: apuntar qué filas van en qué montón. Sin verbo (`.mean()`, `.sum()`, `.size()`) no hay número, y sin número no hay tabla que enseñar.

Por eso sale ese texto feo con una dirección de memoria: ese objeto no sabe dibujarse como tabla, así que Python enseña lo único que sabe decir de él. La dirección cambia en cada ejecución y no significa nada útil.

Y es a propósito: pandas espera a saber qué se le va a pedir para recorrer los datos **una sola vez**. Si calculara la media, la suma y el máximo por si acaso, estaría haciendo trabajo que quizá no quiero.

In [2]:
# groupby agrupa pero si no le decimos que queremos calcular mostrará info del objeto y nada mas
df.groupby("species")

### El objeto sí sabe cosas

Aunque no enseñe datos, por dentro lo tiene todo apuntado: cuántos montones hay, cómo se llaman y quién está en cada uno.

**Ojo al `.shape` de un grupo: `(152, 7)`.** Son 152 filas y **las 7 columnas** de la tabla original. Un grupo no es un resumen: es un trozo entero de la tabla, con todas sus columnas intactas.

In [3]:
# Info de la agrupación para ver en detalle, aun no estamos haciendo calculos

g = df.groupby("species")

print("Número de grupos:", g.ngroups)
print("Nombres de los grupos:", list(g.groups.keys()))
print("Filas del grupo Adelie:", g.get_group("Adelie").shape)

Número de grupos: 3
Nombres de los grupos: ['Adelie', 'Chinstrap', 'Gentoo']
Filas del grupo Adelie: (152, 7)


### ¿Suman las filas de los grupos el total de la tabla?

Aquí sí: 152 + 68 + 124 = 344.

Pero cuadra por suerte del dataset, no porque agrupar sea seguro. Cuadra porque `species` no tiene ni un hueco. La siguiente celda enseña qué pasa cuando sí los hay.

**Esta comprobación parece una tontería y no lo es.** Es la única forma de detectar que se han perdido filas por el camino. Vuelve a aparecer al final del cuaderno, cuando hablamos de ceros falsos.

In [4]:
# Código para comprobar en la agrupación que las filas en total coincidia pero
# tiene trampa porque en esta agrupación no hay valores nulos

print("Adelie:", g.get_group("Adelie").shape[0])
print("Chinstrap:", g.get_group("Chinstrap").shape[0])
print("Gentoo:", g.get_group("Gentoo").shape[0])
print("Total tabla:", len(df))

Adelie: 152
Chinstrap: 68
Gentoo: 124
Total tabla: 344


## Bloque 2 — Error silencioso nº 14: agrupar descarta los nulos

`groupby()` lleva de fábrica el ajuste `dropna=True`. Pandas razona que un hueco no es una categoría, así que no le monta grupo. La consecuencia es que esas filas **desaparecen del cálculo entero**, no solo de un grupo. Y no salta ningún aviso.

Aquí: 165 + 168 = 333, no 344. Faltan 11 pingüinos sin sexo anotado.

Con `dropna=False` aparece un grupo `NaN` con los que faltaban.

**Por qué esto es caro de verdad:** no es que se pierdan filas, es que **el resto de números siguen siendo correctos**. Las medias de cada grupo están bien. Lo que estará mal es todo lo que venga después: el total, los porcentajes, o el "quién vende más" si los que faltan eran todos de uno solo.

> **Regla:** antes de agrupar por una columna, mirar si tiene nulos. Si los tiene, decidir a conciencia: `dropna=False` para verlos, o quitarlos antes dejando escrito por qué.

In [5]:
# Aqui en la agrupación por sexo si vemos que no coincide porque hay nulos 
#  groupby descarta  nulos !!

print("Filas totales:", len(df))
print("Filas por sexo:")
print(df.groupby("sex").size())
df.groupby("sex", dropna=False).size() # El argumento dropna=False incluye nulos y los agrupa

Filas totales: 344
Filas por sexo:
sex
Female    165
Male      168
dtype: int64


sex
Female    165
Male      168
NaN        11
dtype: int64

## Bloque 3 — Sacar un valor concreto del resultado

Dos cosas que mirar en la salida:

**1. Es una `Series`, no un `DataFrame`.** Porque la columna se pidió con el nombre suelto (`["body_mass_g"]`, texto), no dentro de una lista. Igual que en la sesión 3: nombre suelto da columna, lista da tabla.

**2. `species` ya no es una columna.** Se ha convertido en el nombre de cada fila, se ha ido al índice. Por eso `.loc["Gentoo"]` funciona.

### `.loc` frente a `.iloc` aquí

Los dos dan el mismo número hoy. Pero `.iloc[2]` pide "la tercera fila, sea quien sea". El día que entre una especie nueva que ordene antes, devuelve otra cosa y **no da error**.

> **Regla:** para pedir un grupo concreto, siempre `.loc["nombre"]`. Los nombres no se mueven, las posiciones sí.

### Error silencioso nº 15 — el método de más

`media.iloc[2]` ya es un número suelto (`float`). Escribir `media.iloc[2].mean()` funciona y devuelve lo mismo, porque numpy contesta que la media de un solo valor es ese valor.

Pero el código dice una cosa distinta de la que hace: quien lo lea pensará que ahí había una colección de valores, cuando había uno. **Encadenar un método que ya no hace falta sobre un resultado ya calculado** no revienta y no avisa.

In [6]:
# iloc y loc nos da lo mismo pero en este caso agruparemos mejor con loc porque por 
# posición el dia de mañana se pueden añadir mas especies y la posición descuadraría

media= df.groupby("species")["body_mass_g"].mean()
print(media)

media_gentoo= media.iloc[2]
print(media_gentoo)

media_gentoo_b = media.loc["Gentoo"]
print(media_gentoo_b)

species
Adelie       3700.662252
Chinstrap    3733.088235
Gentoo       5076.016260
Name: body_mass_g, dtype: float64
5076.016260162602
5076.016260162602


## Bloque 4 — `size()` y `count()` no son lo mismo

```
.size()                   ->  152, 68, 124
["body_mass_g"].count()   ->  151, 68, 123
```

**Los dos son números correctos. Responden a preguntas distintas.**

- `.size()` cuenta **filas** del grupo. Le da igual lo que haya dentro. Una caja mide lo que mide, esté llena o vacía.
- `.count()` cuenta **valores que existen de verdad** en la columna indicada. Ese sí mira dentro.

### Error silencioso nº 16

Si la pregunta es "cuántos Adelie hay" y se usa `.count()`, la respuesta es 151 y está mal. No porque el código falle, sino porque contestó a otra pregunta. Y el número tiene una pinta estupenda.

**Y su hermano, que es peor:** el `.mean()` del bloque anterior se calculó sobre 151 Adelie, no sobre 152. **Toda operación con números en pandas ignora los huecos por defecto y no lo dice.**

In [7]:
# Los dos son números correctos. Responden a preguntas distintas.

# .size() responde "¿cuántos pingüinos hay de cada especie?"
# .count() sobre body_mass_g responde "¿de cuántos sabemos el peso?"

print(df.groupby("species").size())
print(df.groupby("species")["body_mass_g"].count())


species
Adelie       152
Chinstrap     68
Gentoo       124
dtype: int64
species
Adelie       151
Chinstrap     68
Gentoo       123
Name: body_mass_g, dtype: int64


### Error silencioso nº 17 — elegir columna no siempre cambia nada

`df.groupby("species")["body_mass_g"].size()` devuelve 152, 68, 124. Los mismos números que sin elegir columna.

Lo único que cambia es el `Name: body_mass_g` de abajo: ha arrastrado la etiqueta, pero no ha mirado el contenido. `.size()` cuenta filas y le da igual qué columna se le ponga delante. La columna solo sirvió para poner un título.

**Por qué es peligroso:** el código *parece más específico de lo que es*. Quien lo lea dentro de tres meses verá `["body_mass_g"].size()` y entenderá "pesos que tengo". Y son "pingüinos que hay".

In [8]:
print(df.groupby("species")["body_mass_g"].size())

species
Adelie       152
Chinstrap     68
Gentoo       124
Name: body_mass_g, dtype: int64


### Chequeo de calidad: huecos por grupo

Restar las dos cosas da los huecos de esa columna en cada grupo. Es una comprobación rápida que merece la pena hacer **antes** de fiarse de una media.

**Tip:** pandas resta las dos Series emparejando por **nombre de fila**, no por posición. Por eso funciona aunque el orden fuera distinto. Aquí el índice trabaja a favor.

In [9]:
huecos = df.groupby("species").size() - df.groupby("species")["body_mass_g"].count()
print(huecos)

species
Adelie       1
Chinstrap    0
Gentoo       1
dtype: int64


## Bloque 5 — Varios resúmenes de una vez con `.agg()`

Pedir la media, el máximo y la cuenta en tres celdas distintas no da una tabla, da tres papelitos sueltos. `.agg()` los junta en una tabla con una fila por grupo y una columna por cada cosa pedida.

Fíjate en cómo se le pasan los resúmenes: **una lista de nombres, en texto**. No `.mean()` con paréntesis, sino `"mean"` entre comillas. Es una lista de la compra, no se está ejecutando nada.

### Por qué esto es mejor que tres celdas

Porque **la prueba y el resultado quedan juntos**. La media dice 3700.66 y justo al lado pone 151. No hay forma de mirar la una sin ver la otra.

En tres celdas separadas, la media aparece sola y con toda su pinta de verdad absoluta. Nadie va a subir a buscar de cuántos salió.

> **Costumbre:** cuando pidas medias por grupo, mete siempre `"count"` en la lista. Sale gratis y enseña sobre cuántos está calculada cada una.

In [10]:
resumen = df.groupby("species")["body_mass_g"].agg(["count", "mean", "max"])
print(resumen)
print(type(resumen))

           count         mean     max
species                              
Adelie       151  3700.662252  4775.0
Chinstrap     68  3733.088235  4800.0
Gentoo       123  5076.016260  6300.0
<class 'pandas.core.frame.DataFrame'>


### Lista o suelto, otra vez la misma regla

La única razón de que lo anterior salga `DataFrame` es que se pidieron **tres** cosas.

- `.agg("mean")` → una cosa suelta → **Series**
- `.agg(["mean"])` → lista de una cosa → **DataFrame** de una columna

Es exactamente la misma regla que con los nombres de columna en la sesión 3. El envoltorio manda, no el contenido.

In [11]:
print(type(df.groupby("species")["body_mass_g"].agg("mean")))
print(type(df.groupby("species")["body_mass_g"].agg(["mean"])))

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


### Ponerle nombre propio a las columnas

`count`, `mean` y `max` son nombres de máquina. En un informe quedan feos y, peor, no dicen de qué columna salieron.

Con esta otra forma de escribirlo ya no se elige la columna antes: se le dice a cada resumen por separado en una pareja `(columna, qué hacer)`. Por eso ahora se pueden mezclar peso y aleta en la misma tabla, cosa que antes no se podía.

**Lo que se gana con `n_pesados`:** el nombre de la columna es documentación que viaja con el dato. Va en la tabla, en el CSV que exportes y en el gráfico. `count` se puede leer de dos maneras; `n_pesados` no.

Es lo contrario del error 17: allí el código parecía más específico de lo que era, aquí se le obliga a decir exactamente lo que es.

In [12]:
resumen = df.groupby("species").agg(
    n_pesados=("body_mass_g", "count"),
    peso_medio=("body_mass_g", "mean"),
    aleta_media=("flipper_length_mm", "mean")
)
print(resumen)

           n_pesados   peso_medio  aleta_media
species                                       
Adelie           151  3700.662252   189.953642
Chinstrap         68  3733.088235   195.823529
Gentoo           123  5076.016260   217.186992


### Cada media con la cuenta de su propia columna

En la tabla anterior, `n_pesados` decía sobre cuántos estaba hecho `peso_medio`. Pero **no decía nada sobre `aleta_media`**, que es otra columna con sus propios huecos.

Aquí sí: cada media lleva su cuenta al lado y arriba está el total real del grupo.

`n_pesados` y `n_aletas` coinciden, y por un motivo que conviene entender: los huecos van juntos porque **falta el pingüino entero**, no una medición suelta. Al mismo bicho no lo pesaron ni le midieron la aleta.

Pero cuidado con generalizar: coinciden **en este dataset**. En datos reales pasa lo contrario a menudo (en una encuesta la gente contesta la edad y se salta el sueldo).

> **Regla:** en una tabla de resumen, cada media va acompañada de la cuenta de **su propia** columna. Una cuenta genérica no cubre a las demás.

In [13]:
resumen = df.groupby("species").agg(
    n_total=("species", "size"),
    n_pesados=("body_mass_g", "count"),
    peso_medio=("body_mass_g", "mean"),
    n_aletas=("flipper_length_mm", "count"),
    aleta_media=("flipper_length_mm", "mean")
)
print(resumen)

           n_total  n_pesados   peso_medio  n_aletas  aleta_media
species                                                          
Adelie         152        151  3700.662252       151   189.953642
Chinstrap       68         68  3733.088235        68   195.823529
Gentoo         124        123  5076.016260       123   217.186992


### El `size` de `.agg()` lleva una columna de relleno

Dos columnas idénticas, con columnas de origen distintas. Es el error 17 otra vez: `"size"` cuenta filas del grupo y no mira el contenido de nada.

La sintaxis con nombres **obliga** a escribir una columna en cada pareja, así que para `size` hay que poner una aunque no sirva para nada.

> **Tip:** si escribes `size` en un `.agg()`, pon como columna la misma por la que agrupas. Da igual cuál sea, pero así quien lo lea ve que es de adorno y no se pone a buscar qué tiene de especial esa columna.

In [14]:
print(df.groupby("species").agg(a=("species", "size"), b=("bill_depth_mm", "size")))

             a    b
species            
Adelie     152  152
Chinstrap   68   68
Gentoo     124  124


## Bloque 6 — Agrupar por dos columnas a la vez

Ahora los montones no son "por especie", son "por especie **y** isla". Se pide con una **lista** de columnas.

Salen **5 filas, no 9**. Solo aparecen las combinaciones que **existen de verdad** en los datos: Chinstrap solo vive en Dream y Gentoo solo en Biscoe, así que esas parejas no llegan a existir.

### El índice ahora es una lista de parejas

Hasta ahora cada fila se llamaba de una sola manera: `Adelie`. Ahora una fila es "Adelie **en** Torgersen": hacen falta **dos datos** para señalar una sola fila.

En la salida se ve la columna de la izquierda partida en dos, y la especie no se repite en cada línea, solo se escribe cuando cambia. Es pandas ahorrando tinta: el valor está en todas las filas aunque no se vea escrito.

In [15]:
por_especie_isla = df.groupby(["species", "island"]).size()
print(por_especie_isla)

species    island   
Adelie     Biscoe        44
           Dream         56
           Torgersen     52
Chinstrap  Dream         68
Gentoo     Biscoe       124
dtype: int64


### Cómo se pide con dos niveles

- `.loc["Adelie"]` → un nombre → devuelve **un trozo** (todas las islas de Adelie)
- `.loc[("Adelie", "Torgersen")]` → una pareja → devuelve **un número**

**Detalle importante:** al pedir solo `"Adelie"`, en la salida pone `island` arriba. Pandas ha **quitado** el nivel de especie porque ya no hace falta, y devuelve una Series normal indexada por isla. La pareja se ha vuelto un nombre suelto.

In [16]:
print(por_especie_isla.loc["Adelie"])
print(por_especie_isla.loc[("Adelie", "Torgersen")])

island
Biscoe       44
Dream        56
Torgersen    52
dtype: int64
52


### Error silencioso nº 18 — las combinaciones que no salen

Las cuatro combinaciones que faltan **no valen cero. Simplemente no están.**

Para un humano "no hay Gentoo en Torgersen" y "hay 0 Gentoo en Torgersen" son la misma frase. Para el código no: en un gráfico de barras no se ve una barra a cero, se ve que ese hueco no existe. Si la tabla fuera ventas por producto y mes, los meses sin ventas desaparecen y la serie temporal tiene agujeros que nadie ve.

### `observed`: qué hacer con las que faltan

| Valor | Qué hace |
|---|---|
| `observed=True` | Solo las combinaciones **vistas** en los datos |
| `observed=False` | **Todas las posibles**, con 0 en las que no aparecen |

En la celda siguiente salen `(5,)` y `(5,)`: no hizo nada. El motivo está justo debajo.

**Nota sobre `.shape`:** en una Series devuelve `(5,)`, con la coma suelta, porque solo tiene una dimensión. En un DataFrame devuelve dos números, como aquel `(152, 7)`.

In [17]:
print(df.groupby(["species", "island"]).size().shape)
print(df.groupby(["species", "island"], observed=False).size().shape)

(5,)
(5,)


### Por qué `observed` no hizo nada

Porque necesita saber cuáles son "todas las posibles", y una columna de texto no lo sabe.

**La analogía:**

- **Texto (`object`)** es un montón de papelitos escritos a mano. Para saber qué islas hay solo se puede leer los papelitos y apuntar los distintos. Lo que hay es lo que se ha escrito.
- **Categoría (`category`)** es un formulario con desplegable. Las opciones están metidas **por dentro del propio tipo**, en una lista aparte. Aunque nadie haya elegido "Torgersen", la opción sigue en el desplegable.

Esa lista aparte (el **catálogo**) es la diferencia entera. `observed=False` significa "sácame también las opciones del catálogo que nadie ha elegido". Sin catálogo, no hay nada que sacar.

Convirtiendo las dos columnas a `category`, ahora sí: 5 y 9.

> **Tip:** el valor por defecto de `observed` ha ido cambiando entre versiones de pandas. Si el resultado importa, escríbelo a mano en vez de fiarte de lo que venga puesto.

In [19]:
print(df["species"].dtype, df["island"].dtype)

df2 = df.copy()
df2["species"] = df2["species"].astype("category")
df2["island"] = df2["island"].astype("category")

print(df2.groupby(["species", "island"], observed=True).size().shape)
print(df2.groupby(["species", "island"], observed=False).size().shape)

object object
(5,)
(9,)


## Bloque 7 — El catálogo sobrevive al filtro

Aquí es donde `category` gana de verdad. Los Chinstrap solo viven en Dream.

**Con texto:** una sola fila, Dream. En ese trozo de tabla las otras dos islas **no existen**.

**Con categoría:** tres filas, con Biscoe y Torgersen a cero.

In [21]:
chinstrap_txt = df[df["species"] == "Chinstrap"]
print(chinstrap_txt.groupby("island", observed=False).size())

island
Dream    68
dtype: int64


In [20]:
df_cat = df.copy()
df_cat["island"] = df_cat["island"].astype("category")

chinstrap_cat = df_cat[df_cat["species"] == "Chinstrap"]

print(chinstrap_cat["island"].cat.categories)
print(chinstrap_cat.groupby("island", observed=False).size())

Index(['Biscoe', 'Dream', 'Torgersen'], dtype='object')
island
Biscoe        0
Dream        68
Torgersen     0
dtype: int64


### Por qué el texto "se olvida"

La pregunta natural es: si al mirar `df` se ven tres islas, ¿por qué al agrupar el trozo solo sale una?

**Porque en la tabla que se está agrupando ya no hay tres islas. Hay una.**

`df[df["species"] == "Chinstrap"]` no marca unas filas de `df`: crea **una tabla nueva** de 68 filas. En la columna `island` de esa tabla nueva, la palabra "Biscoe" no aparece ni una sola vez. No es que pandas no quiera verla: no está escrita en ningún sitio.

**Una tabla filtrada no guarda ningún recuerdo de la tabla de la que salió.** Es un objeto independiente. Ese vínculo está en la cabeza del analista, no en los datos.

Es la misma familia que lo de la sesión 3, cuando el índice conservaba los números de fila originales pero la posición ya no correspondía. El filtro corta, y lo que queda fuera deja de existir para todo lo que venga después.

Y por eso `category` sirve: **el catálogo va pegado a la columna, no a la tabla**. Al filtrar, las filas se van pero el catálogo viaja con la columna.

In [23]:
print(df["island"].unique())
print(chinstrap_txt["island"].unique())

['Torgersen' 'Biscoe' 'Dream']
['Dream']


## Bloque 8 — Error silencioso nº 19: el orden de las operaciones

`astype("category")` construye el catálogo **mirando lo que hay dentro en ese momento**.

- **Categorizar tarde** (después de filtrar): mira 68 filas donde solo pone Dream y monta un catálogo de un valor. El paso se ha dado, pero no se ha ganado nada.
- **Categorizar pronto** (antes de filtrar): el catálogo se arma con las tres islas y el filtro se lleva las filas pero no toca el catálogo.

Mismos datos, mismo código, distinto orden, distinto resultado. **Y ninguna de las dos versiones da error.**

### El orden en un flujo de verdad

1. Cargar los datos
2. **Arreglar tipos**: fechas a fecha, categorías a categoría
3. Limpiar (nulos, duplicados)
4. Filtrar
5. Agrupar y resumir

La categorización va en el paso 2, junto a las fechas. Es una decisión sobre **cómo son los datos**, no sobre qué análisis se va a hacer.

> **Regla general:** cualquier cosa que dependa del conjunto completo hay que hacerla mientras todavía se tiene el conjunto completo.

> **Tip:** categoriza cuando la columna tenga pocos valores distintos y sean un conjunto cerrado: islas, especies, provincias, estados de un pedido. No categorices nombres, direcciones ni IDs, que son casi todos distintos y no se gana nada.

In [26]:
tarde = df[df["species"] == "Chinstrap"].copy()
tarde["island"] = tarde["island"].astype("category")
print("Categorizado tarde:", list(tarde["island"].cat.categories))

print("Categorizado pronto:", list(chinstrap_cat["island"].cat.categories))

Categorizado tarde: ['Dream']
Categorizado pronto: ['Biscoe', 'Dream', 'Torgersen']


## Bloque 9 — El cero verdadero y el cero mentiroso

Con el catálogo bien montado ya salen los ceros: Biscoe 0, Dream 68, Torgersen 0.

Aquí esos ceros son **verdad**: son islas que existen, que se visitaron, donde se buscaron pingüinos y no había Chinstrap. Cero es un hallazgo real.

Pero en el trabajo real casi nunca es tan limpio. Imagina una tabla de ventas por tienda y mes, y una casilla que pone "Tienda Norte, marzo: 0".

| El cero dice | Lo que pasó de verdad |
|---|---|
| 0 ventas | Abrió, atendió, no vendió nada. **Verdad.** |
| 0 ventas | Vendió, pero esas filas se cayeron por el camino. **Mentira.** |
| 0 ventas | La tienda no existía ese mes. **No aplica.** |

Los tres se escriben igual en la casilla. **Un `0` no lleva etiqueta.**

### Qué quiere decir "se cayeron por el camino"

Que desaparecieron antes de llegar a la cuenta. Formas de que pase:

- **Un nulo.** En 500 filas nadie rellenó el campo "tienda". `groupby` no sabe en qué montón meterlas y las tira del cálculo entero (error 14).
- **Un filtro anterior.** Se filtró por `importe > 0` para quitar devoluciones y de paso se fueron ventas legítimas de importe cero.
- **Un nombre mal escrito.** En unas filas pone "Tienda Norte" y en otras "T. Norte". Para pandas son dos tiendas distintas.
- **No llegaron.** La tienda mandó el fichero tarde y ese mes no se cargó.

En los cuatro casos la casilla queda vacía, se rellena con cero, y el cero es mentira. Las ventas ocurrieron y el dinero entró; lo que falló fue la anotación.

### Error silencioso nº 20

Casi siempre el paso siguiente es hacer una media, y esos ceros entran como si fueran meses malos:

- Si el cero es real, tiene que entrar.
- Si es un fallo de datos, hunde la media con una venta que sí existió.
- Si la tienda no existía, hunde la media con un mes que no debería contar.

Mismo número, tres tratamientos distintos, y la media sale bien redonda en los tres casos.

> **Regla:** rellenar huecos con cero **no es neutral**. Es afirmar que hubo medición y dio cero. Antes de poner un cero hay que saber si se midió.

**Cómo se distingue:** no se puede, mirando la tabla de resumen. Hay que volver atrás y comprobar cuántas filas tenía el original, cuántas quedaron después de agrupar, y si cuadra. Es exactamente la comprobación del bloque 1, la de 152 + 68 + 124 = 344, que entonces parecía una tontería.

# Conclusiones de la sesión 4

## Qué verbo uso

| Quiero saber | Escribo |
|---|---|
| Cuántas filas hay en cada grupo | `.size()` |
| De cuántas sé el dato de esa columna | `["columna"].count()` |
| Cuántos huecos tiene esa columna en cada grupo | `.size() - ["columna"].count()` |
| La media de esa columna por grupo | `["columna"].mean()` |
| Varias cosas de golpe, con nombre propio | `.agg(nombre=("columna", "verbo"))` |
| Un grupo concreto entero | `.get_group("nombre")` |
| Un valor del resultado | `.loc["nombre"]` o `.loc[("nombre", "otro")]` |

## Errores silenciosos añadidos en esta sesión

*(La numeración va seguida desde las sesiones anteriores, que cerraron en el 13.)*

| nº | Qué pasa |
|---|---|
| 14 | `groupby()` descarta las filas con nulo en la columna de agrupar, sin avisar |
| 15 | Encadenar un método que ya no hace falta (`.mean()` sobre un número) funciona y engaña |
| 16 | `.count()` y `.size()` dan números creíbles, pero contestan preguntas distintas |
| 17 | Elegir columna antes de `.size()` no cambia el resultado, solo la etiqueta |
| 18 | Las combinaciones que no existen no salen: faltan, no valen cero |
| 19 | El orden de las operaciones cambia el resultado sin que ninguna dé error |
| 20 | Rellenar huecos con cero afirma que hubo medición; puede ser falso |

## Lo que hay que llevarse

1. Agrupar es **dividir, calcular y juntar**. Sin verbo no hay cálculo.
2. La columna por la que agrupo **se va al índice**: se pide con `.loc["nombre"]`, nunca por posición.
3. **Los nulos se caen solos en todas partes**: al agrupar, al contar y al hacer medias. Nunca avisan.
4. **Toda media va acompañada de su cuenta.** Un promedio sin saber sobre cuántos está hecho no es un dato, es una impresión.
5. **Una tabla filtrada no recuerda de dónde salió.** Lo que dependa del conjunto completo, se hace antes de filtrar.
6. **Un cero puede ser un hallazgo o un fallo**, y se escriben igual.
7. Antes de fiarse de un resumen por grupos, **comprobar que las filas suman el total**.

## Pendiente para la próxima

Queda un bloque de esta sesión sin ver: **por qué la media de las medias no es la media**. Es el error más caro de la familia y ya está todo lo necesario para entenderlo.